In [23]:
import os
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [24]:
spark = SparkSession.builder.appName('HousingModelDocker').getOrCreate()

## Step 1: Read CSV using Spark

In [25]:
data = spark.read.csv('work/kc_house_data.csv', header=True, inferSchema=True)
data.printSchema()


root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)



In [26]:
data.show()

+----------+---------------+---------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|    price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+---------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000| 221900.0|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
|6414100192|20141209T000000| 538000.0|       3|     2.25|       2570|    7242|   2.0|         0|   0|        3|    7|      2170|          40

In [27]:
(
    data.select
        ([
            F.count(
                F.when(
                    F.col(c).isNull(), 1
                )
            ).alias(c) for c in data.columns
        ]).show()
)

+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+
| id|date|price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat|long|sqft_living15|sqft_lot15|
+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+
|  0|   0|    0|       0|        0|          0|       0|     0|         0|   0|        0|    0|         0|            0|       0|           0|      0|  0|   0|            0|         0|
+---+----+-----+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+---+----+-------------+----------+



In [28]:
data = data.drop('id','date','zipcode')

## Step 2: Split Dataset into train and test data

In [29]:
training_data, testing_data = data.randomSplit([0.8, 0.2], seed=42)
print(f'Training data: {training_data.count()}')
print(f'Testing data: {testing_data.count()}')


Training data: 17349
Testing data: 4264


## Step 3: Create feature Vector and pipeline

In [30]:
training_data.printSchema()

root
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)



In [31]:
numerical_columns = [
    'bedrooms',
    'bathrooms',
    'sqft_living',
    'sqft_lot',
    'floors',
    'waterfront',
    'view',
    'condition',
    'grade',
    'yr_built',
    'yr_renovated',
    'sqft_above',
    'sqft_basement',
    'sqft_living15',
    'sqft_lot15'
]

assembler = VectorAssembler(inputCols=numerical_columns, outputCol='unscaled_feature')
scaler = StandardScaler(inputCol='unscaled_feature', outputCol='features', withMean=True, withStd=True)
lr = LinearRegression(featuresCol='features', labelCol='price',regParam=0.001)


## Step 4: Train a model

In [32]:
pipeline = Pipeline(stages=[assembler, scaler, lr])
pipeline_model = pipeline.fit(training_data)
prediction_data = pipeline_model.transform(testing_data)

## Step 5: Evaluate your model

In [33]:
evaluator = RegressionEvaluator(labelCol='price',predictionCol='prediction', metricName='mae')
mae = evaluator.evaluate(prediction_data)
print(f'MAE: {mae}')

MAE: 137634.54471146225
